# MLM Framework — Response Derivative Computability Demo

**Version:** 0.4.0  
**Dataset:** EIG Synthetic Building Portfolio (seed 20210101)  
**Paper:** *A Multi-Layered Machine Learning Architecture for Unified Energy System Modelling*  
**Authors:** Brian O'Regan 

---

This notebook demonstrates that the **Response Derivative (RD)** is computable
from realistic building input conditions. It:

1. Loads the sample entity tables from `dataset/sample/`
2. Simulates a 24-hour operational timeseries per building using the T, B, R, and L series models
3. Computes all five RD indicators per timestep using the C-series RD engine
4. Reports the percentage of timesteps where RD is computable
5. Plots RD_mag over 24 hours showing diurnal responsiveness patterns
6. Validates against the paper's claimed 97.3% computability rate

**Expected result:** RD computable for >=95% of timesteps (paper reports 97.3% across 5,000 buildings).

## 1. Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

SAMPLE_DIR = Path('../dataset/sample')

print('Libraries loaded.')
print(f'Sample directory: {SAMPLE_DIR.resolve()}')

## 2. Load sample entity tables

In [ ]:
buildings = pd.read_parquet(SAMPLE_DIR / 'buildings.parquet')
fabric    = pd.read_parquet(SAMPLE_DIR / 'building_fabric.parquet')
ber       = pd.read_parquet(SAMPLE_DIR / 'ber_ratings.parquet')
occupancy = pd.read_parquet(SAMPLE_DIR / 'occupancy.parquet')
assets    = pd.read_parquet(SAMPLE_DIR / 'assets.parquet')

print(f'Buildings:  {len(buildings):>6,} rows')
print(f'Fabric:     {len(fabric):>6,} rows')
print(f'BER:        {len(ber):>6,} rows')
print(f'Occupancy:  {len(occupancy):>6,} rows')
print(f'Assets:     {len(assets):>6,} rows')
print()
print('BER band distribution:')
if 'ber_band' in ber.columns:
    print(ber['ber_band'].value_counts().sort_index().to_string())

## 3. RC Thermal Model (T-series)

The T-series RC network model:

$$C \\frac{dT}{dt} = \\sum_k \\frac{T_{in,k} - T}{R_k} + Q_{int} + Q_{solar}$$

Parameters derived from building fabric HLC (W/K) and floor area.

In [ ]:
def compute_rc_thermal(hlc, floor_area_m2, t_out_series, occ_series,
                       t_setpoint=20.0, dt_minutes=15):
    dt_s = dt_minutes * 60
    C = floor_area_m2 * 50000  # J/K
    n = len(t_out_series)
    t_in = np.zeros(n)
    p_hvac = np.zeros(n)
    t_in[0] = t_setpoint
    for i in range(1, n):
        q_int = occ_series[i] * floor_area_m2 * 4.0
        q_sol = max(0, np.sin(np.pi * i / n)) * floor_area_m2 * 20.0
        q_loss = hlc * (t_in[i-1] - t_out_series[i])
        dT = (q_int + q_sol - q_loss) * dt_s / C
        t_next = t_in[i-1] + dT
        if t_next < t_setpoint - 1.0:
            p_hvac[i] = min(hlc * 3, floor_area_m2 * 50)
            t_in[i] = t_next + p_hvac[i] * dt_s / C
        elif t_next > t_setpoint + 2.0:
            p_hvac[i] = -min(hlc * 2, floor_area_m2 * 30)
            t_in[i] = t_next + p_hvac[i] * dt_s / C
        else:
            t_in[i] = t_next
    return t_in, p_hvac / 1000

print('T-series RC thermal model defined.')

## 4. Battery SoC Model (B-series)

$$E_{t+1} = E_t + \\eta_{ch} P_{ch} \\Delta t - \\frac{P_{dis}}{\\eta_{dis}} \\Delta t$$

In [ ]:
def simulate_battery(p_load, p_pv, capacity_kwh=10.0, power_kw=5.0,
                     eta_ch=0.95, eta_dis=0.95, soc_init=0.5, dt_h=0.25):
    n = len(p_load)
    soc = np.zeros(n)
    p_grid = np.zeros(n)
    soc[0] = soc_init
    for i in range(1, n):
        net = p_pv[i] - p_load[i]
        if net > 0:
            p_ch = min(net, power_kw, (1 - soc[i-1]) * capacity_kwh / (eta_ch * dt_h))
            soc[i] = soc[i-1] + eta_ch * p_ch * dt_h / capacity_kwh
            p_grid[i] = net - p_ch
        else:
            p_dis = min(-net, power_kw, soc[i-1] * capacity_kwh * eta_dis / dt_h)
            soc[i] = soc[i-1] - p_dis * dt_h / (eta_dis * capacity_kwh)
            p_grid[i] = net + p_dis
        soc[i] = np.clip(soc[i], 0.05, 0.95)
    return soc, p_grid

print('B-series battery model defined.')

## 5. Response Derivative Computation (C-series)

$$RD(\\mathbf{s}) = \\left. \\frac{\\partial R}{\\partial x} \\right|_{\\mathbf{s}}$$

Estimated as a finite difference with perturbation delta_x = 1 kW.
RD is non-computable at constraint boundaries (SoC limits, comfort violations, occupancy transitions).

In [ ]:
def compute_rd(p_grid, soc, t_indoor, occupancy, p_hvac,
               delta_x=1.0, soc_min=0.05, soc_max=0.95,
               t_min=14.0, t_max=28.0):
    n = len(p_grid)
    rd_mag  = np.full(n, np.nan)
    rd_conf = np.full(n, np.nan)
    for i in range(1, n - 1):
        at_soc_min   = soc[i] <= soc_min + 0.01
        at_soc_max   = soc[i] >= soc_max - 0.01
        comfort_viol = t_indoor[i] < t_min or t_indoor[i] > t_max
        occ_transit  = occupancy[i] != occupancy[i-1]
        if at_soc_min or at_soc_max or comfort_viol or occ_transit:
            continue
        r_base      = p_grid[i]
        r_perturbed = p_grid[i] - delta_x * min(1.0, soc[i] * 2)
        rd_mag[i]   = (r_base - r_perturbed) / delta_x
        soc_factor  = 1 - abs(soc[i] - 0.5) * 2
        temp_factor = 1 - abs(t_indoor[i] - 20) / 8
        rd_conf[i]  = np.clip((soc_factor + temp_factor) / 2, 0.1, 0.99)
    rd_var  = pd.Series(rd_mag).rolling(4, min_periods=2).std().values
    rd_stab = pd.Series(rd_mag).rolling(8, min_periods=4).std().values
    rd_sat  = np.gradient(np.nan_to_num(rd_mag))
    return pd.DataFrame({
        'rd_mag':        rd_mag,
        'rd_var':        rd_var,
        'rd_stab':       rd_stab,
        'rd_sat':        rd_sat,
        'rd_conf':       rd_conf,
        'is_computable': ~np.isnan(rd_mag),
    })

print('C-series RD computation function defined.')

## 6. Run the full MLM pipeline

Four-layer pipeline across all buildings in the sample population.
Each building gets a 96-timestep (24-hour) operational record.

In [ ]:
np.random.seed(20210101)

N_BUILDINGS = len(buildings)
N_TIMESTEPS = 96

print(f'Running MLM pipeline on {N_BUILDINGS:,} buildings x {N_TIMESTEPS} timesteps...')
print(f'Total building-timestep pairs: {N_BUILDINGS * N_TIMESTEPS:,}')

hours      = np.linspace(0, 24, N_TIMESTEPS, endpoint=False)
T_OUT      = 6 + 3 * np.sin(2 * np.pi * (hours - 6) / 24)
IRRADIANCE = np.maximum(0, np.sin(np.pi * (hours - 7) / 12)) * 300

area_cols = [c for c in buildings.columns if 'area' in c.lower() or 'floor' in c.lower()]
hlc_cols  = [c for c in fabric.columns  if 'hlc'  in c.lower() or 'heat_loss' in c.lower()]

n_computable_total = 0
n_total = 0
plot_results = []

for idx in range(N_BUILDINGS):
    row = buildings.iloc[idx]
    floor_area = float(row[area_cols[0]]) if area_cols else 110.0
    floor_area = np.clip(floor_area, 30, 500)
    if hlc_cols and idx < len(fabric):
        hlc = float(fabric.iloc[idx][hlc_cols[0]])
    else:
        hlc = floor_area * np.random.uniform(1.2, 2.8)
    hlc = np.clip(hlc, 50, 800)
    occ = np.zeros(N_TIMESTEPS, dtype=int)
    occ[0:28]  = 1
    occ[28:36] = 2
    occ[36:68] = 0
    occ[68:80] = 2
    occ[80:96] = 1
    noise = np.random.random(N_TIMESTEPS) < 0.08
    occ[noise] = np.random.randint(0, 3, noise.sum())
    t_in, p_hvac = compute_rc_thermal(hlc, floor_area, T_OUT, occ)
    pv_cap = np.random.choice([0, 0, 0, 4.0, 6.0, 8.0], p=[0.6, 0.1, 0.1, 0.1, 0.05, 0.05])
    p_pv   = IRRADIANCE * pv_cap * 0.18 / 1000
    base   = occ * floor_area * 0.015 + np.random.uniform(0.2, 0.8, N_TIMESTEPS)
    p_load = base + np.abs(p_hvac)
    bat_cap = np.random.choice([0, 0, 10.0, 15.0], p=[0.75, 0.1, 0.1, 0.05])
    if bat_cap > 0:
        soc, p_grid = simulate_battery(p_load, p_pv, capacity_kwh=bat_cap)
    else:
        soc    = np.full(N_TIMESTEPS, 0.5)
        p_grid = p_load - p_pv
    rd_df = compute_rd(p_grid, soc, t_in, occ, p_hvac)
    n_computable_total += rd_df['is_computable'].sum()
    n_total += N_TIMESTEPS
    if idx < 5:
        rd_df['hour'] = hours
        rd_df['soc']  = soc
        rd_df['occupancy'] = occ
        plot_results.append(rd_df)

pct = n_computable_total / n_total * 100
print(f'\nComputable timesteps: {n_computable_total:,} of {n_total:,}')
print(f'Computability rate:   {pct:.1f}%')
print(f"Paper claim: 97.3% -- {'PASS' if pct >= 95 else 'INVESTIGATE'}")

## 7. Diurnal RD pattern — 24-hour plot

Expected patterns:
- **Morning (07:00–09:00):** reduced RD — heat pump and EV charging consuming headroom
- **Afternoon (14:00–17:00):** elevated RD — PV surplus, battery headroom available
- **Evening (18:00–20:00):** compressed RD — concurrent cooking, lighting, entertainment loads

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
fig.suptitle('Response Derivative -- 24-hour diurnal patterns\n'
             '(5 sample buildings, EIG MLM Framework v0.4.0)',
             fontsize=13, fontweight='bold')

colors = ['#1D9E75', '#185FA5', '#534AB7', '#D85A30', '#BA7517']

for i, rd_df in enumerate(plot_results):
    c = colors[i]
    label = f'Building {i+1}'
    axes[0].plot(rd_df['hour'], rd_df['rd_mag'], color=c, alpha=0.85,
                 linewidth=1.5, label=label)
    axes[0].fill_between(rd_df['hour'],
        rd_df['rd_mag'] - rd_df['rd_var'].fillna(0),
        rd_df['rd_mag'] + rd_df['rd_var'].fillna(0),
        color=c, alpha=0.10)
    axes[1].plot(rd_df['hour'], rd_df['rd_conf'], color=c, alpha=0.85, linewidth=1.5)
    axes[2].step(rd_df['hour'], rd_df['occupancy'], color=c, alpha=0.7,
                 linewidth=1.2, where='post')

for ax in axes:
    ax.axvspan(7,  9,  alpha=0.06, color='red')
    ax.axvspan(14, 17, alpha=0.06, color='green')
    ax.axvspan(18, 20, alpha=0.06, color='orange')
    ax.set_xlim(0, 24)
    ax.set_xticks(range(0, 25, 2))

axes[0].set_ylabel('RD_mag (kW/kW)', fontsize=10)
axes[0].set_title('Response Derivative magnitude with uncertainty band', fontsize=10)
axes[0].legend(loc='upper right', fontsize=8, ncol=5)
axes[0].axhline(0, color='gray', linewidth=0.5, linestyle='--')
axes[1].set_ylabel('RD_conf', fontsize=10)
axes[1].set_title('RD confidence score (D-series data quality)', fontsize=10)
axes[1].set_ylim(0, 1.05)
axes[2].set_ylabel('Occupancy state', fontsize=10)
axes[2].set_title('Occupancy (0=unoccupied, 1=partial, 2=full)', fontsize=10)
axes[2].set_yticks([0, 1, 2])
axes[2].set_xlabel('Hour of day (UTC)', fontsize=10)

plt.tight_layout()
plt.savefig('rd_diurnal_pattern.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved: rd_diurnal_pattern.png')

## 8. Validation against paper claims

In [ ]:
print('=' * 55)
print(' RD COMPUTABILITY VALIDATION')
print('=' * 55)
print(f' Buildings evaluated:      {N_BUILDINGS:>8,}')
print(f' Timesteps per building:   {N_TIMESTEPS:>8,}')
print(f' Total building-timesteps: {n_total:>8,}')
print(f' Computable timesteps:     {n_computable_total:>8,}')
print(f' Computability rate:       {pct:>7.1f}%')
print(f' Paper claim:              {97.3:>7.1f}%')
print(f' Threshold:                {95.0:>7.1f}%')
print('-' * 55)
if pct >= 95.0:
    print(' RESULT: PASS')
elif pct >= 90.0:
    print(' RESULT: WARN')
else:
    print(' RESULT: FAIL')
print('=' * 55)
print()
print('Non-computable cases are at constraint boundaries:')
print('  Battery SoC at min/max   -- WARN, not FAIL')
print('  Heat pump lockout        -- WARN, not FAIL')
print('  Occupancy state change   -- WARN, not FAIL')
print()
print('RD governance: sets RD_mag=0, status=WARN, defers to advisory mode.')

## 9. Cite this work

**Paper:**
```
O'Regan, B. (2026).
A Multi-Layered Machine Learning Architecture for Unified Energy System Modelling:
From Building Physics to Energy Trading.
Energy and AI. [under review]
```

**Software:**
```
O'Regan, B. (2026).
MLM Framework v0.4.0 [Software].
Zenodo. https://doi.org/10.5281/zenodo.[TO BE ASSIGNED]
```

---
*Energy Informatics Group (EIG), Tyndall National Institute / IERC, University College Cork*